[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iamjamesbowden/AG983/blob/main/class5/AG983_Week10_Workshop.ipynb)

# AG983 | Workshop Five: The Analyst's Edge

**Textual Analytics for Accounting and Finance**  
University of Strathclyde Business School


<div style="background:#0f172a;border:2px solid #818cf8;border-radius:10px;padding:24px 30px;font-family:Georgia,serif;color:#e2e8f0;">
<h2 style="color:#818cf8;margin:0 0 6px 0;"> Before you start &#8212; two-minute setup</h2>
<p style="color:#94a3b8;margin:0 0 18px 0;font-size:0.9em;">Complete both steps below before running any cells. You only need to do this once.</p>

<div style="background:#1e293b;border-radius:8px;padding:16px 20px;margin-bottom:14px;">
<p style="color:#f59e0b;font-weight:bold;margin:0 0 12px 0;">&#128193; Step 1 &nbsp;&mdash;&nbsp; Add the data folder to your Google Drive</p>
<ol style="color:#e2e8f0;font-size:0.92em;line-height:2.1;margin:0;padding-left:20px;">
  <li>Open this link in a new tab:<br/>
      <code style="background:#0f172a;padding:4px 10px;border-radius:4px;color:#60a5fa;font-size:0.95em;">
      https://drive.google.com/drive/folders/1Nf0X7aCBce8knRfyN7N-B1ErSoioxj7N</code></li>
  <li>Near the top of the page, click the <strong>small arrow</strong> next to the <strong>clips</strong> folder name</li>
  <li>Select <strong>Organise</strong>, then <strong>&#8220;Add shortcut to Drive&#8221;</strong></li>
  <li>Choose <strong>My Drive</strong> (the very top level &#8212; do <em>not</em> place it inside any subfolder)</li>
  <li>Click <strong>Add shortcut</strong></li>
</ol>
<p style="color:#94a3b8;font-size:0.84em;margin:10px 0 0 0;">&#10003; &nbsp;You should now see a folder called <strong style="color:#e2e8f0;">clips</strong> in your Google Drive root. You do not need to open it or move anything inside it.</p>
</div>

<div style="background:#1e293b;border-radius:8px;padding:16px 20px;">
<p style="color:#22c55e;font-weight:bold;margin:0 0 12px 0;">&#9654; Step 2 &nbsp;&mdash;&nbsp; Run the setup cells</p>
<ol style="color:#e2e8f0;font-size:0.92em;line-height:2.1;margin:0;padding-left:20px;">
  <li>Click on the <strong>&#9654; Setup</strong> cell directly below this box and press <strong>Shift+Enter</strong> (or click the run button &#9654;)</li>
  <li>Google will ask you to grant Drive access &#8212; click <strong>Connect to Google Drive</strong> and follow the prompts</li>
  <li>You should see: <code style="background:#0f172a;padding:2px 8px;border-radius:3px;color:#22c55e;">Drive mounted. Data folder found.</code></li>
  <li>Then run the <strong>Load data</strong> cell and the <strong>Utilities</strong> cell</li>
  <li>After all three cells show a green tick, scroll down to the workshop</li>
</ol>
<p style="color:#94a3b8;font-size:0.84em;margin:10px 0 0 0;">&#9888; &nbsp;If you see <strong style="color:#f59e0b;">Drive folder not found</strong>, you placed the shortcut inside a subfolder. Go back to Step 1 and make sure you select <em>My Drive</em> at the top level, not a nested folder.</p>
</div>

</div>

In [ ]:
# @title ⚙ Setup — run first {display-mode: "form"}
import subprocess, sys, io, contextlib, warnings
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'ipywidgets', 'plotly', 'vaderSentiment', '--quiet'], capture_output=True)
warnings.filterwarnings('ignore')

from google.colab import drive
# Suppress the "already mounted" info message — it is harmless but confusing
_buf = io.StringIO()
with contextlib.redirect_stdout(_buf):
    drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import glob

# Auto-discover the data folder by searching for workshop_manifest.json
# Works regardless of what the student named the shortcut or where they placed it
_hits = glob.glob('/content/drive/MyDrive/**/workshop_manifest.json', recursive=True)
if _hits:
    DRIVE_ROOT = str(Path(_hits[0]).parent)
    print(f'✓  Drive mounted. Data folder found at: {DRIVE_ROOT}')
else:
    DRIVE_ROOT = '/content/drive/MyDrive/clips'   # fallback
    print('⚠  workshop_manifest.json not found under My Drive.')
    print('   Complete Step 1 in the setup box above, then re-run this cell.')
    print('   (If you just added the shortcut, try Runtime → Restart and run all.)')


In [ ]:
# @title 📦 Load data {display-mode: "form"}
import json, re, os, urllib.request
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML, Image, clear_output
from pathlib import Path

with open(f'{DRIVE_ROOT}/workshop_manifest.json') as _f:
    MANIFEST = json.load(_f)

CALLS    = MANIFEST['calls']
ENDPOINT = MANIFEST['google_apps_script_endpoint']
QUARTERS = [c['quarter'] for c in CALLS]

def dpath(fname): return f'{DRIVE_ROOT}/{fname}'
IMAGES_ROOT = f'{DRIVE_ROOT}/images'

RETURNS    = pd.read_csv(dpath(MANIFEST['files']['returns']))
GAAP       = pd.read_csv(dpath(MANIFEST['files']['gaap_metrics']))
KPI        = pd.read_csv(dpath(MANIFEST['files']['nonfin_kpis']))
GRAND_MEAN = pd.read_csv(dpath(MANIFEST['files']['grand_mean_final'])).iloc[0]
SUMMARY    = pd.read_csv(dpath(MANIFEST['files']['all_calls_summary']))

FEATURES = {}
IMG_MFST = {}
for _call in CALLS:
    _q = _call['quarter']
    FEATURES[_q] = pd.read_csv(dpath(_call['files']['windowed_features']))
    IMG_MFST[_q] = pd.read_csv(dpath(_call['files']['image_manifest']))

CLR = {
    'lm_positive':  '#22c55e',
    'lm_negative':  '#ef4444',
    'lm_neutral':   '#9ca3af',
    'fou_above':    '#f59e0b',
    'fou_below':    '#3b82f6',
    'ret_positive': '#22c55e',
    'ret_negative': '#ef4444',
    'analyst_text': '#fbbf24',
    'musk_text':    '#60a5fa',
}

STATE = {
    'team_name': '', 'registered': False,
    'r1_submitted': False, 'r2_submitted': False, 'r3_submitted': False,
    'power_play_quarter': None,
    'chips': {q: 3 for q in QUARTERS},
    'twitter_chips': 0,
    'twitter_score': 0,
}

# Anonymous labels (non-chronological — prevents chart cross-referencing)
ANON_MAP = {
    "2022Q4": "Call A",
    "2023Q1": "Call B",
    "2024Q1": "Call C",
    "2024Q2": "Call D",
}
ANON_QUARTERS = [ANON_MAP[q] for q in QUARTERS]
print('Data loaded.')


In [ ]:
# @title 🔧 Utilities {display-mode: "form"}
display(HTML("<style>\n  .ag-card{background:#1e293b;border:1px solid #334155;border-radius:8px;padding:16px;margin:8px;font-family:'Georgia',serif;color:#e2e8f0;}\n  .ag-card h4{color:#818cf8;margin:0 0 8px 0;font-size:1.1em;}\n  .ag-excerpt-box{background:#0f172a;border-left:3px solid #334155;padding:10px 14px;border-radius:4px;font-size:0.88em;line-height:1.6;margin:8px 0;}\n  .ag-analyst{color:#fbbf24;}\n  .ag-musk{color:#60a5fa;}\n  .ag-label{color:#94a3b8;font-size:0.8em;text-transform:uppercase;letter-spacing:0.05em;margin-bottom:2px;}\n  .ag-gauge-pos{color:#22c55e;font-size:1.4em;font-weight:bold;}\n  .ag-gauge-neg{color:#ef4444;font-size:1.4em;font-weight:bold;}\n  .ag-gauge-neu{color:#9ca3af;font-size:1.4em;font-weight:bold;}\n  .ag-section-hdr{background:linear-gradient(90deg,#1e293b,#0f172a);border-left:4px solid #818cf8;padding:10px 16px;margin:20px 0 10px 0;font-family:'Georgia',serif;color:#e2e8f0;font-size:1.1em;}\n  .ag-status-box{background:#1e293b;border:1px solid #334155;border-radius:6px;padding:12px;font-family:monospace;font-size:0.85em;color:#94a3b8;}\n  .ag-table{border-collapse:collapse;width:100%;}\n  .ag-table th{background:#334155;color:#e2e8f0;padding:6px 10px;text-align:left;font-size:0.85em;}\n  .ag-table td{padding:5px 10px;font-size:0.85em;border-bottom:1px solid #1e293b;color:#e2e8f0;}\n  .ag-table tr:nth-child(even){background:#1e293b;}\n  .ag-table tr:nth-child(odd){background:#0f172a;}\n  .ag-arrow-up{color:#22c55e;font-weight:bold;}\n  .ag-arrow-down{color:#ef4444;font-weight:bold;}\n  .ag-arrow-flat{color:#9ca3af;}\n  .ag-na{color:#475569;font-style:italic;}\n  .ag-word-counter{font-size:0.78em;color:#94a3b8;text-align:right;margin-top:2px;}\n  .widget-radio-box label{color:#e2e8f0 !important;}\n  .widget-radio-box .widget-label-basic{color:#e2e8f0 !important;}\n  .widget-readout{color:#e2e8f0 !important;}\n  .widget-label{color:#94a3b8 !important;}\n  .pp-box{background:#1e293b;border:1px solid #f59e0b;border-radius:8px;padding:14px 18px;margin:10px 0;}\n</style>\n"))

def post_to_script(payload):
    if not ENDPOINT or 'ENDPOINT_URL_HERE' in ENDPOINT:
        print('Apps Script endpoint not configured (offline mode).')
        return {'success': True, 'message': 'offline'}
    import requests as _req, json as _j
    r = _req.post(ENDPOINT, json=payload, timeout=20,
                  headers={'Content-Type': 'application/json'})
    return _j.loads(r.text)

def get_status(round_number):
    try: return post_to_script({'action':'get_status','round':round_number})
    except: return {'submitted_teams':[],'total':'?'}

def render_sentiment_bars(ax, sentences, title=''):
    nets  = [s['net'] for s in sentences]
    cats  = [s['category'] for s in sentences]
    colors = [CLR['lm_positive'] if c=='positive' else
              CLR['lm_negative'] if c=='negative' else CLR['lm_neutral']
              for c in cats]
    ax.bar(range(len(sentences)), nets, color=colors, width=0.7)
    ax.axhline(0, color='#475569', linewidth=0.8)
    ax.set_xticks(range(len(sentences)))
    ax.set_xticklabels([f'S{i+1}' for i in range(len(sentences))],
                       fontsize=7, color='#94a3b8')
    ax.set_ylabel('LM net', fontsize=7, color='#94a3b8')
    ax.tick_params(axis='y', labelsize=7, colors='#94a3b8')
    ax.set_facecolor('#0f172a')
    for sp in ['top','right']: ax.spines[sp].set_visible(False)
    for sp in ['bottom','left']: ax.spines[sp].set_color('#334155')
    if title: ax.set_title(title, fontsize=8, color='#818cf8', pad=4)

FOU_MEAN = float(GRAND_MEAN['fraction_of_unvoiced_mean'])

RADAR_FEATS  = ['mean_pitch','mean_intensity','number_of_periods',
                'fraction_of_unvoiced','number_of_voice_breaks',
                'jitter_local','shimmer_local','mean_autocorrelation','mean_nhr']
RADAR_LABELS = ['Pitch level','Intensity','Speech rate','Unvoiced ratio',
                'Voice breaks','Jitter (freq)','Shimmer (amp)',
                'Autocorrelation','HNR (breathiness)']
RADAR_HOVER  = [
    'Mean fundamental frequency — higher = more animated delivery',
    'Loudness envelope — higher = more emphatic speech',
    'Proportion of voiced frames — lower = more pausing',
    'Fraction of unvoiced frames — higher = more silence/hesitation',
    'Abrupt voicing interruptions — higher = more speech disfluency',
    'Cycle-to-cycle pitch variation — higher = less stable pitch',
    'Cycle-to-cycle amplitude variation — higher = less stable volume',
    'Short-term periodicity — higher = cleaner, more periodic voice',
    'Noise-to-harmonics ratio — higher = breathier, more noise in voice',
]

def z_score(val, feat):
    mu  = GRAND_MEAN.get(f'{feat}_mean', np.nan)
    sig = GRAND_MEAN.get(f'{feat}_std',  np.nan)
    if not sig or np.isnan(sig): return 0.0
    return (val - mu) / sig

def make_silence_map(c):
    q = c['quarter']
    df = FEATURES[q].copy()
    kq = c['key_question']['window_index']
    kq_text = (c['key_question']['analyst_text_preview'] or '')[:80]
    colors = [CLR['fou_above'] if v > FOU_MEAN else CLR['fou_below']
              for v in df['fraction_of_unvoiced']]
    fig = go.Figure()
    fig.add_trace(go.Bar(x=df['window_index'], y=df['fraction_of_unvoiced'],
        marker_color=colors,
        hovertemplate='<b>Window %{x}</b><br>FoU: %{y:.3f}<extra></extra>'))
    fig.add_hline(y=FOU_MEAN, line_dash='dot', line_color='#94a3b8',
        annotation_text=f'Mean ({FOU_MEAN:.3f})',
        annotation_font_color='#94a3b8', annotation_font_size=10)
    if kq is not None and kq < len(df):
        fig.add_vline(x=kq, line_dash='dash', line_color='#818cf8',
            annotation_text='Key Q', annotation_font_color='#818cf8',
            annotation_font_size=10)
    fig.update_layout(
        title=dict(text=f'{ANON_MAP.get(q,q)} — Silence patterns across the Q&A',
                   font=dict(color='#e2e8f0', size=13)),
        xaxis=dict(title='5-min window', color='#94a3b8', gridcolor='#1e293b'),
        yaxis=dict(title='Fraction of unvoiced (0–1)', range=[0,1],
                   color='#94a3b8', gridcolor='#1e293b'),
        plot_bgcolor='#0f172a', paper_bgcolor='#1e293b',
        font=dict(color='#e2e8f0'), height=260,
        margin=dict(l=50,r=20,t=50,b=40), showlegend=False)
    return fig

def make_radar(c):
    q = c['quarter']
    row = SUMMARY[SUMMARY['call'] == f'TSLA_{q}']
    if len(row) == 0: return go.Figure()
    row = row.iloc[0]
    MAX_Z = 3.0
    def norm(z): return max(0, min(1, (z + MAX_Z) / (2 * MAX_Z)))
    r_call = []
    flags  = []
    for feat in RADAR_FEATS:
        val = row.get(f'{feat}_mean_musk', np.nan)
        if pd.isna(val): val = 0.0
        z = z_score(val, feat)
        r_call.append(norm(z))
        flags.append(abs(z) > 1.0)
    r_grand = [0.5] * len(RADAR_FEATS)
    labels  = RADAR_LABELS
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=r_call+[r_call[0]], theta=labels+[labels[0]],
        fill='toself', fillcolor='rgba(96,165,250,0.15)',
        line=dict(color='#60a5fa', width=2), name=ANON_MAP.get(q,q)))
    fig.add_trace(go.Scatterpolar(r=r_grand+[r_grand[0]], theta=labels+[labels[0]],
        fill='none', line=dict(color='#fbbf24', width=2, dash='dot'),
        name='Cross-call average'))
    for i,(feat,flag) in enumerate(zip(RADAR_FEATS, flags)):
        if flag:
            fig.add_trace(go.Scatterpolar(r=[r_call[i]], theta=[labels[i]],
                mode='markers', marker=dict(color='#f59e0b', size=10),
                showlegend=False))
    fig.update_layout(
        polar=dict(bgcolor='#0f172a',
            radialaxis=dict(visible=True, range=[0,1], color='#475569',
                            gridcolor='#334155', tickfont_size=7),
            angularaxis=dict(color='#94a3b8', gridcolor='#334155')),
        showlegend=True,
        legend=dict(font=dict(color='#94a3b8', size=9), bgcolor='rgba(0,0,0,0)'),
        paper_bgcolor='#1e293b',
        title=dict(text=f'{ANON_MAP.get(q,q)} — Feature radar (Musk responses)',
                   font=dict(color='#e2e8f0', size=12)),
        height=320, margin=dict(l=40,r=40,t=50,b=20))
    return fig

def arrow_cell(val, yoy, unavail=False):
    if unavail or (isinstance(val, float) and np.isnan(val)):
        return '<span class="ag-na">n/a</span>'
    yoy_str = ''
    if yoy is not None and not (isinstance(yoy, float) and np.isnan(yoy)):
        if yoy > 2:
            yoy_str = f' <span class="ag-arrow-up">↑{yoy:.1f}%</span>'
        elif yoy < -2:
            yoy_str = f' <span class="ag-arrow-down">↓{abs(yoy):.1f}%</span>'
        else:
            yoy_str = ' <span class="ag-arrow-flat">→</span>'
    return f'{val:,.0f}{yoy_str}'

def _c(row, col, yoy_col):
    v = row.get(col)
    unavail = bool(row.get(f'{col}_unavailable', pd.isna(v) if v is not None else True))
    if pd.isna(v): return '<span class="ag-na">n/a</span>'
    yoy = row.get(yoy_col)
    return arrow_cell(v, yoy if (yoy is not None and not pd.isna(yoy)) else None, unavail)

def build_gaap_table(fy):
    rows = GAAP[GAAP['fy'] == fy]
    if len(rows) == 0: return '<p>No GAAP data for this year.</p>'
    r = rows.iloc[0].to_dict()
    gm = r.get('gross_margin_pct','')
    gm_str = f'{gm:.1f}%' if isinstance(gm, float) and not np.isnan(gm) else 'n/a'
    body = (
        f'<tr><td>Total Revenue (USD m)</td><td>{_c(r,"total_revenue","total_revenue_yoy_pct")}</td></tr>'
        f'<tr><td>Gross Profit (USD m)</td><td>{_c(r,"gross_profit","gross_margin_pct_yoy_pct")}</td></tr>'
        f'<tr><td>Gross Margin %</td><td>{gm_str}</td></tr>'
        f'<tr><td>Operating Income (USD m)</td><td>{_c(r,"operating_income","operating_income_yoy_pct")}</td></tr>'
        f'<tr><td>Net Income (USD m)</td><td>{_c(r,"net_income","net_income_yoy_pct")}</td></tr>'
        f'<tr><td>EPS Diluted</td><td>{_c(r,"eps_diluted","eps_diluted_yoy_pct")}</td></tr>'
    )
    return (f'<table class="ag-table"><tr><th>GAAP Metric</th>'
            f'<th>Value (YoY)</th></tr>{body}</table>')

def build_kpi_table(fy):
    rows = KPI[KPI['fy'] == fy]
    if len(rows) == 0: return '<p>No KPI data.</p>'
    r = rows.iloc[0].to_dict()
    sc = '<span class="ag-na">n/a — not reported in 10-K</span>'
    body = (
        f'<tr><td>Vehicle Deliveries</td><td>{_c(r,"vehicle_deliveries","vehicle_deliveries_yoy_pct")}</td></tr>'
        f'<tr><td>Vehicle Production</td><td>{_c(r,"vehicle_production","vehicle_production_yoy_pct")}</td></tr>'
        f'<tr><td>Energy Storage (GWh)</td><td>{_c(r,"energy_storage_gwh","energy_storage_gwh_yoy_pct")}</td></tr>'
        f'<tr><td>Solar Deployed (MW)</td><td>{_c(r,"solar_deployed_mw","solar_deployed_mw_yoy_pct")}</td></tr>'
        f'<tr><td>Supercharger Stations</td><td>{sc}</td></tr>'
        f'<tr><td>Supercharger Connectors</td><td>{sc}</td></tr>'
    )
    return (f'<table class="ag-table"><tr><th>Non-financial KPI</th>'
            f'<th>Value (YoY)</th></tr>{body}</table>')

def _resolve_img(raw_path, fy=None):
    """Try Drive-relative paths when local path doesn't exist."""
    for candidate in [
        raw_path,
        str(Path(IMAGES_ROOT) / f'TSLA_{fy}' / Path(raw_path).name) if fy else None,
        str(Path(IMAGES_ROOT) / Path(raw_path).name),
    ]:
        if candidate and Path(candidate).exists():
            return candidate
    return None

def make_radar_key():
    rows = ''.join(
        f'<tr><td style="color:#60a5fa;padding:3px 8px;font-size:0.8em;">{lbl}</td>'
        f'<td style="color:#94a3b8;padding:3px 8px;font-size:0.78em;">{tip}</td></tr>'
        for lbl, tip in zip(RADAR_LABELS, RADAR_HOVER)
    )
    return (f'<details style="margin:6px 0;"><summary style="color:#475569;'
            f'font-size:0.8em;cursor:pointer;">What do the radar axes mean? '
            f'(Todd et al., 2025)</summary>'
            f'<table style="border-collapse:collapse;margin-top:6px;">'
            f'{rows}</table></details>')

print('Utilities ready.')


<div style='background:#0f172a;padding:28px 32px;border-radius:10px;border:1px solid #334155;font-family:Georgia,serif;color:#e2e8f0;'>
<h1 style='color:#818cf8;margin:0 0 6px 0;font-size:1.8em;'>The Analyst&#8217;s Edge</h1>
<h2 style='color:#94a3b8;margin:0 0 20px 0;font-weight:normal;font-size:1.1em;'>Multimodal Analysis of Tesla Earnings Calls &nbsp;&middot;&nbsp; AG983 Workshop Five</h2>
<p style='line-height:1.8;margin-bottom:14px;'>It is the morning after a Tesla earnings call. You are a buy-side analyst at a mid-size asset manager. Your portfolio manager is waiting for your note before the market opens. You have three sources of evidence: the text of the Q&amp;A session, a set of infographics from the most recent annual report, and acoustic measurements of Elon Musk&#8217;s voice during the call. Your task is to integrate these signals and predict how Tesla&#8217;s stock will perform over the next five trading days.</p>
<p style='line-height:1.8;margin-bottom:14px;'>You will work through <strong style='color:#e2e8f0;'>three rounds</strong>, each introducing one new modality. In each round you set or update your prediction and provide a brief rationale. At the end the actual outcomes are revealed and the team with the highest score wins the session.</p>
<p style='line-height:1.8;color:#94a3b8;font-size:0.92em;margin:0 0 16px 0;'><strong style='color:#e2e8f0;'>Four earnings calls</strong> have been selected from the Tesla transcript archive (2022&#8211;2024), labelled <strong style='color:#818cf8;'>Call A&#8211;D</strong> in randomised order. You will not know which real-world quarter each label corresponds to until the Reveal at the end &#8212; this prevents you looking up the answer on the price chart.</p>

<div style='background:#1e293b;border:1px solid #334155;border-radius:8px;padding:18px 22px;margin-bottom:14px;'>
<p style='color:#818cf8;font-weight:bold;margin:0 0 12px 0;font-size:1em;'>&#127922; How chips work</p>
<p style='line-height:1.8;margin:0 0 10px 0;'>In Round 1 you allocate <strong>1, 2, or 3 chips</strong> to each call (12 total). Think of chips as confidence: the more chips you place on a call, the more you gain if you are right &#8212; and the more you lose if you are wrong. After viewing the infographics (Round 2) and the audio analysis (Round 3) you get two further chances to <strong>shift chips</strong> between calls &#8212; you may raise one call to 4 chips by taking 1 from another, keeping the total at 12.</p>
<div style='background:#0f172a;border-radius:6px;padding:12px 16px;margin-bottom:10px;'>
<table style='border-collapse:collapse;width:100%;font-size:0.88em;'>
<tr><td style='color:#94a3b8;padding:3px 12px 3px 0;'>Correct prediction</td><td style='color:#22c55e;font-weight:bold;'>+chips points</td></tr>
<tr><td style='color:#94a3b8;padding:3px 12px 3px 0;'>Wrong prediction</td><td style='color:#ef4444;font-weight:bold;'>&#8722;chips points</td></tr>
</table>
</div>
<p style='line-height:1.8;margin:0 0 6px 0;color:#94a3b8;font-size:0.88em;'><strong style='color:#e2e8f0;'>Example:</strong> you put 3 chips on Call A and predict &#8220;Stock rose&#8221;. If it did rise, you score +3. If it fell, you score &#8722;3.</p>
</div>

<div style='background:#1e1b4b;border:1px solid #818cf8;border-radius:8px;padding:18px 22px;margin-bottom:4px;'>
<p style='color:#f59e0b;font-weight:bold;margin:0 0 10px 0;font-size:1em;'>&#9889; The Power Play &#8212; your one wild card</p>
<p style='line-height:1.8;margin:0 0 10px 0;'>After Round 3 you may (optionally) declare a <strong>Power Play</strong> on exactly one call. On that call your chips count <strong style='color:#f59e0b;'>double</strong> in the final score &#8212; but a wrong call also costs double. You can only do this once, and it is irreversible.</p>
<div style='background:#0f172a;border-radius:6px;padding:12px 16px;margin-bottom:10px;'>
<table style='border-collapse:collapse;width:100%;font-size:0.88em;'>
<tr><td style='color:#94a3b8;padding:3px 12px 3px 0;'>Power Play call correct</td><td style='color:#f59e0b;font-weight:bold;'>+chips &times; 2</td></tr>
<tr><td style='color:#94a3b8;padding:3px 12px 3px 0;'>Power Play call wrong</td><td style='color:#ef4444;font-weight:bold;'>&#8722;chips &times; 2</td></tr>
<tr><td style='color:#94a3b8;padding:3px 12px 3px 0;'>No Power Play selected</td><td style='color:#94a3b8;'>all calls score at &times;1</td></tr>
</table>
</div>
<p style='line-height:1.8;margin:0;color:#94a3b8;font-size:0.88em;'><strong style='color:#e2e8f0;'>Example:</strong> you staked 3 chips on Call B and declare it your Power Play. If Call B is correct you score +6. If wrong, you score &#8722;6. Choose the call you feel most confident about after Round 3.</p>
</div>

</div>

In [ ]:
# @title 📋 Step 1 — Register your team {display-mode: "form"}
display(HTML('<div class="ag-section-hdr">Step 1 of 4 &nbsp;&middot;&nbsp; Register your team</div>'))
team_input = widgets.Text(placeholder='e.g. Team Hawkins',
    description='Team name:', layout=widgets.Layout(width='340px'),
    style={'description_width': '90px'})
reg_btn    = widgets.Button(description='Register', button_style='primary',
    layout=widgets.Layout(width='120px'))
reg_status = widgets.HTML()

def on_register(b):
    name = team_input.value.strip()
    if not name:
        reg_status.value = '<span style="color:#ef4444;">Please enter a team name.</span>'
        return
    STATE['team_name'] = name
    try:
        resp = post_to_script({'action':'register_team','team_name':name})
        msg = 'already registered' if resp.get('message')=='already_registered' else 'registered'
        if resp.get('success'):
            STATE['registered'] = True
            reg_status.value = (f'<span style="color:#22c55e;">✓ <strong>{name}</strong>'
                                f' {msg}. You are ready to begin.</span>')
            team_input.disabled = True; reg_btn.disabled = True
        else:
            reg_status.value = f'<span style="color:#ef4444;">Registration failed.</span>'
    except Exception:
        STATE['registered'] = True
        reg_status.value = (f'<span style="color:#f59e0b;">⚠ Offline mode — '
                            f'team name saved locally as <strong>{name}</strong>.</span>')

reg_btn.on_click(on_register)
display(widgets.HBox([team_input, reg_btn]))
display(reg_status)


In [ ]:
# @title 📈 Tesla stock price context {display-mode: "form"}
display(HTML('<div class="ag-section-hdr">Tesla stock price 2021&#8211;2024</div>'))
display(HTML("<div style='background:#1e293b;border:1px solid #334155;border-radius:8px;padding:16px 22px;margin:12px 0;font-family:Georgia,serif;color:#e2e8f0;line-height:1.8;'><p style='margin:0 0 10px 0;'>The chart below shows Tesla's daily closing share price from January 2021 to December 2024. Each dashed vertical line marks the trading day of a Tesla earnings call. The lines are colour-coded by outcome: <span style='color:#22c55e;font-weight:bold;'>green</span> indicates that Tesla's stock <em>outperformed</em> the S&amp;P 500 over the five trading days following that call (positive abnormal return); <span style='color:#ef4444;font-weight:bold;'>red</span> indicates that it <em>underperformed</em> the S&amp;P 500 over the same window (negative abnormal return).</p><p style='margin:0 0 10px 0;'><strong>Abnormal return</strong> is defined here as the cumulative TSLA return over days +1 to +5 after the call, minus the cumulative S&amp;P 500 return over the same window. It isolates the market&#8217;s reaction to the call itself from broader market moves on those days. A positive abnormal return means the call was net good news for Tesla relative to the market; a negative abnormal return means it was net bad news.</p><p style='margin:0 0 6px 0;'>Four of these calls have been selected for today&#8217;s workshop. For each one, you will receive three sources of evidence in sequence &#8212; the text of the Q&amp;A, financial infographics, and acoustic measurements of Elon Musk&#8217;s voice &#8212; and must predict whether that call&#8217;s line would be <span style='color:#22c55e;font-weight:bold;'>green</span> or <span style='color:#ef4444;font-weight:bold;'>red</span> on this chart. The outcomes are locked and hidden until every team has submitted all three rounds.</p></div>\n"))
try:
    display(Image(filename=f'{DRIVE_ROOT}/TSLA_price_timeline_workshop.png', width=900))
except Exception:
    display(HTML('<p style="color:#94a3b8;">Timeline image not found.</p>'))
display(HTML("<div style='background:#1e293b;border:1px solid #334155;border-radius:8px;padding:14px 20px;margin:12px 0;font-family:Georgia,serif;'>\n<p style='color:#818cf8;margin:0 0 10px 0;font-weight:bold;'>Colour grammar &#8212; consistent throughout this notebook</p>\n<table style='border-collapse:collapse;'>\n<tr><td style='padding:4px 16px 4px 0;'><span style='background:#22c55e;padding:3px 10px;border-radius:4px;color:#fff;font-size:0.85em;'>&#9632; Green</span></td><td style='color:#e2e8f0;font-size:0.88em;padding:4px 0;'>LM positive sentiment &nbsp;|&nbsp; Positive stock return</td></tr>\n<tr><td style='padding:4px 16px 4px 0;'><span style='background:#ef4444;padding:3px 10px;border-radius:4px;color:#fff;font-size:0.85em;'>&#9632; Red</span></td><td style='color:#e2e8f0;font-size:0.88em;'>LM negative sentiment &nbsp;|&nbsp; Negative stock return</td></tr>\n<tr><td style='padding:4px 16px 4px 0;'><span style='background:#9ca3af;padding:3px 10px;border-radius:4px;color:#fff;font-size:0.85em;'>&#9632; Grey</span></td><td style='color:#e2e8f0;font-size:0.88em;'>LM neutral sentiment</td></tr>\n<tr><td style='padding:4px 16px 4px 0;'><span style='background:#f59e0b;padding:3px 10px;border-radius:4px;color:#fff;font-size:0.85em;'>&#9632; Amber</span></td><td style='color:#e2e8f0;font-size:0.88em;'>Above-average fraction of unvoiced (more silence / hesitation)</td></tr>\n<tr><td style='padding:4px 16px 4px 0;'><span style='background:#3b82f6;padding:3px 10px;border-radius:4px;color:#fff;font-size:0.85em;'>&#9632; Blue</span></td><td style='color:#e2e8f0;font-size:0.88em;'>Below-average fraction of unvoiced (more voiced / confident)</td></tr>\n</table>\n<p style='color:#94a3b8;font-size:0.82em;margin:10px 0 0 0;'><strong style='color:#e2e8f0;'>Chip allocation:</strong> Allocate 1, 2, or 3 chips per call before submitting Round 1 — no fixed total required. Higher allocation amplifies your score for that call. You may shift one chip once in Round 2 and once in Round 3.</p>\n</div>\n"))


<div style='background:linear-gradient(90deg,#1e1b4b,#0f172a);padding:20px 26px;border-radius:8px;border-left:4px solid #818cf8;margin:20px 0 12px 0;font-family:Georgia,serif;'>
<div style='display:flex;align-items:baseline;gap:12px;margin-bottom:8px;'>
  <h2 style='color:#818cf8;margin:0;'>Round 1 &#8212; The Text</h2>
  <span style='color:#475569;font-size:0.85em;'>Step 1 of 3</span>
</div>
<p style='color:#e2e8f0;margin:0 0 12px 0;font-size:0.95em;line-height:1.7;'>
  For each of the four calls below you will see the highest-signal question&#8211;answer pair from the transcript Q&amp;A session. Read the analyst&#8217;s question and Elon Musk&#8217;s response carefully.
</p>
<div style='background:rgba(0,0,0,0.3);border-radius:6px;padding:12px 16px;margin-bottom:12px;'>
<p style='color:#818cf8;font-size:0.82em;font-weight:bold;text-transform:uppercase;letter-spacing:0.05em;margin:0 0 8px 0;'>What to look for</p>
<ul style='color:#94a3b8;font-size:0.88em;line-height:2;margin:0;padding-left:18px;'>
  <li>Does Musk sound <strong style='color:#e2e8f0;'>confident and specific</strong>, or <strong style='color:#e2e8f0;'>vague and defensive</strong>?</li>
  <li>Are there concrete numbers and forward-looking commitments, or mostly hedging language?</li>
  <li>Check the <strong style='color:#e2e8f0;'>LM sentiment score</strong> at the bottom of each excerpt &#8212; positive means net optimistic language, negative means net pessimistic.</li>
  <li>The sentence tints (<span style='background:rgba(34,197,94,0.3);padding:1px 6px;border-radius:3px;'>green</span> / <span style='background:rgba(239,68,68,0.3);padding:1px 6px;border-radius:3px;'>red</span>) highlight individual sentences flagged by the Loughran&#8211;McDonald financial dictionary.</li>
</ul>
</div>
<div style='background:rgba(0,0,0,0.3);border-radius:6px;padding:12px 16px;'>
<p style='color:#818cf8;font-size:0.82em;font-weight:bold;text-transform:uppercase;letter-spacing:0.05em;margin:0 0 8px 0;'>Then for each call</p>
<ol style='color:#94a3b8;font-size:0.88em;line-height:2;margin:0;padding-left:18px;'>
  <li>Set your <strong style='color:#e2e8f0;'>chip allocation</strong> (1&#8211;3). Be honest about your confidence &#8212; you cannot change chips after Round 1.</li>
  <li>Select <strong style='color:#e2e8f0;'>Stock rose</strong> or <strong style='color:#e2e8f0;'>Stock fell</strong>.</li>
  <li>Write a <strong style='color:#e2e8f0;'>rationale</strong> in max 30 words &#8212; cite specific evidence from the text, not just a gut feel.</li>
</ol>
</div>
</div>

In [ ]:
# @title 🔧 Round helpers {display-mode: "form"}
# ── helpers ──────────────────────────────────────────────────────────────────
def _fmt_sentences(sentences, base_col):
    """Render sentences with subtle LM sentiment tinting."""
    parts = []
    for s in sentences:
        txt = s["text"].strip()
        if not txt:
            continue
        cat = s.get("category", "neutral")
        if cat == "positive":
            bg, border = "rgba(34,197,94,0.15)", "#22c55e"
        elif cat == "negative":
            bg, border = "rgba(239,68,68,0.15)", "#ef4444"
        else:
            bg, border = "transparent", "transparent"
        parts.append(
            f'<span style="background:{bg};border-bottom:1px solid {border};' +
            f'border-radius:2px;padding:0 2px;">{txt}</span> '
        )
    return "".join(parts)

def make_qa_html(c):
    exc   = c["excerpt"]
    q     = c["quarter"]
    anon  = ANON_MAP.get(q, q)
    a_html = _fmt_sentences(exc["analyst_sentences"], "#fbbf24")
    m_html = _fmt_sentences(exc["musk_sentences"],    "#60a5fa")
    net    = exc["musk_net_sent"]
    if net > 0.005:
        gcol, gsym, glbl = "#22c55e", f"+{net:.3f}", "net positive"
    elif net < -0.005:
        gcol, gsym, glbl = "#ef4444", f"{net:.3f}",  "net negative"
    else:
        gcol, gsym, glbl = "#9ca3af", f"{net:.3f}",  "near-neutral"
    return (
        f'<div style="background:#1e293b;border:1px solid #334155;border-radius:8px;' +
        f'padding:16px;font-family:Georgia,serif;color:#e2e8f0;">' +
        f'<h4 style="color:#818cf8;margin:0 0 4px 0;">{anon}</h4>' +
        f'<div style="color:#94a3b8;font-size:0.75em;text-transform:uppercase;' +
        f'letter-spacing:0.05em;margin-bottom:8px;">Analyst: {exc["analyst_speaker"]}</div>' +
        f'<div style="background:#0f172a;border-left:3px solid #fbbf24;border-radius:4px;' +
        f'padding:10px 14px;margin-bottom:10px;max-height:160px;overflow-y:auto;">' +
        f'<div style="color:#fbbf24;font-size:0.72em;text-transform:uppercase;' +
        f'letter-spacing:0.07em;margin-bottom:6px;">Analyst question</div>' +
        f'<div style="font-size:0.88em;line-height:1.8;color:#e2e8f0;">{a_html}</div></div>' +
        f'<div style="background:#0f172a;border-left:3px solid #60a5fa;border-radius:4px;' +
        f'padding:10px 14px;margin-bottom:10px;max-height:280px;overflow-y:auto;">' +
        f'<div style="color:#60a5fa;font-size:0.72em;text-transform:uppercase;' +
        f'letter-spacing:0.07em;margin-bottom:6px;">Elon Musk response</div>' +
        f'<div style="font-size:0.88em;line-height:1.8;color:#e2e8f0;">{m_html}</div></div>' +
        f'<div style="font-size:0.82em;margin-top:4px;">' +
        f'<span style="color:#94a3b8;">Aggregate LM sentiment (Musk):&nbsp;</span>' +
        f'<span style="color:{gcol};font-weight:bold;">{gsym}</span>' +
        f'&nbsp;<span style="color:#94a3b8;">({glbl})</span>' +
        f'<span style="color:#475569;font-size:0.85em;">&nbsp;&nbsp;Sentence tints: ' +
        f'<span style="background:rgba(34,197,94,0.25);padding:1px 5px;border-radius:2px;">positive</span> ' +
        f'<span style="background:rgba(239,68,68,0.25);padding:1px 5px;border-radius:2px;">negative</span> ' +
        f'no tint = neutral</span></div></div>'
    )

def make_sent_chart(c):
    out = widgets.Output()
    with out:
        fig, axes = plt.subplots(1, 2, figsize=(7, 2.0))
        fig.patch.set_facecolor('#1e293b')
        render_sentiment_bars(axes[0], c["excerpt"]["analyst_sentences"], "Analyst sentences")
        render_sentiment_bars(axes[1], c["excerpt"]["musk_sentences"],    "Musk sentences")
        plt.tight_layout(pad=0.4); plt.show(); plt.close()
    return out

# ── per-call input widgets ────────────────────────────────────────────────────
chip_sliders   = {}
pred_dropdowns = {}
reason_areas   = {}
word_counters  = {}

for _call in CALLS:
    _q = _call["quarter"]
    chip_sliders[_q] = widgets.IntSlider(
        value=3, min=1, max=3, step=1,
        description="Chips:", style={"description_width": "55px"},
        layout=widgets.Layout(width="260px"))
    pred_dropdowns[_q] = widgets.Dropdown(
        options=["-- select --", "Stock rose", "Stock fell"],
        description="Predict:", style={"description_width": "55px"},
        layout=widgets.Layout(width="230px"))
    reason_areas[_q] = widgets.Textarea(
        placeholder="Your rationale (max 30 words) …",
        layout=widgets.Layout(width="100%", height="90px"))
    word_counters[_q] = widgets.HTML(
        value='<div class="ag-word-counter">0 / 30 words</div>')
    def _mk_wc(_q_=_q):
        def _upd(ch):
            wc  = len(ch["new"].split())
            col = "#ef4444" if wc > 30 else "#94a3b8"
            word_counters[_q_].value = (
                f'<div class="ag-word-counter" style="color:{col};">{wc} / 30 words</div>')
        return _upd
    reason_areas[_q].observe(_mk_wc(), names="value")

# ── combined side-by-side rows ────────────────────────────────────────────────
call_rows = []
for _call in CALLS:
    _q = _call["quarter"]
    left_panel = widgets.VBox(
        [widgets.HTML(value=make_qa_html(_call)), make_sent_chart(_call)],
        layout=widgets.Layout(width="58%", min_width="0"))
    right_panel = widgets.VBox(
        [widgets.HTML(value=(
             f'<div style="font-family:Georgia;color:#818cf8;font-weight:bold;' +
             f'font-size:1.05em;margin-bottom:12px;">{ANON_MAP.get(_q,_q)}</div>')),
         chip_sliders[_q],
         pred_dropdowns[_q],
         widgets.HTML(value=(
             '<div style="color:#94a3b8;font-size:0.85em;font-family:Georgia;' +
             'margin:10px 0 4px 0;">Rationale <span style="color:#475569;">' +
             '(max 30 words)</span></div>')),
         reason_areas[_q],
         word_counters[_q]],
        layout=widgets.Layout(
            width="40%", min_width="0", padding="16px",
            border="1px solid #334155", border_radius="8px"))
    call_rows.append(widgets.HBox(
        [left_panel, right_panel],
        layout=widgets.Layout(margin="0 0 20px 0", align_items="flex-start",
                              grid_gap="16px")))

display(widgets.VBox(call_rows))


In [ ]:
# @title 🔵 Round 1 — Directional predictions {display-mode: "form"}
display(HTML('<div class="ag-section-hdr">Round 1 &#8212; Submit your predictions</div>'))


r1_submit_btn = widgets.Button(description="Submit Round 1",
    button_style="success", layout=widgets.Layout(width="200px", height="38px"))
r1_status     = widgets.HTML()
r1_status_box = widgets.Output()

def on_r1_submit(b):
    if not STATE.get("team_name"):
        r1_status.value = '<span style="color:#ef4444;">Register your team first.</span>'; return
    if STATE.get("r1_submitted"):
        r1_status.value = '<span style="color:#f59e0b;">Already submitted.</span>'; return
    for q in QUARTERS:
        if pred_dropdowns[q].value == "-- select --":
            r1_status.value = f'<span style="color:#ef4444;">Select prediction for {ANON_MAP.get(q,q)}.</span>'; return
        wc = len(reason_areas[q].value.split())
        if wc == 0:
            r1_status.value = f'<span style="color:#ef4444;">Enter rationale for {ANON_MAP.get(q,q)}.</span>'; return
        if wc > 30:
            r1_status.value = f'<span style="color:#ef4444;">{ANON_MAP.get(q,q)}: rationale over 30 words ({wc}).</span>'; return
    STATE["chips"] = {q: chip_sliders[q].value for q in QUARTERS}
    payload = {"action": "submit_round1", "team_name": STATE["team_name"],
               "calls": [{"quarter": q, "chips": chip_sliders[q].value,
                          "prediction": pred_dropdowns[q].value,
                          "reasoning": reason_areas[q].value.strip()}
                         for q in QUARTERS]}
    try:
        resp = post_to_script(payload)
        if not resp.get("success"):
            r1_status.value = f'<span style="color:#ef4444;">Error: {resp.get("message")}</span>'; return
    except Exception:
        pass
    STATE["r1_submitted"] = True
    STATE["r1_predictions"] = {q: pred_dropdowns[q].value for q in QUARTERS}
    # Live-update R2 and R3 prediction banners
    try:
        for _bq in QUARTERS:
            _bpred = STATE["r1_predictions"][_bq]
            _bchip = STATE["chips"].get(_bq, 3)
            _bpc   = '#22c55e' if 'rose' in _bpred.lower() else '#ef4444'
            _bhtml = (
                f'<div style="background:#0f172a;border-left:3px solid {_bpc};'
                f'padding:8px 12px;margin:0 0 10px 0;border-radius:0 4px 4px 0;'
                f'font-family:Georgia,serif;font-size:0.84em;">'
                f'<span style="color:#475569;">Your Round\u00a01 call: </span>'
                f'<b style="color:{_bpc};">{_bpred}</b>'
                f'<span style="color:#475569;"> \u2014 {_bchip} chips</span></div>')
            if 'r2_banners' in dir(): r2_banners[_bq].value = _bhtml
    except Exception:
        pass
    for q in QUARTERS:
        chip_sliders[q].disabled = True; pred_dropdowns[q].disabled = True
        reason_areas[q].disabled = True
    r1_submit_btn.disabled = True
    r1_status.value = ('<span style="color:#22c55e;">&#10003; Round 1 submitted. '
        'Wait for the facilitator to unlock Round 2 &#8212; you will see a count below.</span>')
    with r1_status_box:
        try:
            st  = get_status(1)
            sub = st.get("submitted_teams", [])
            display(HTML(
                f'<div class="ag-status-box">Teams submitted: ' +
                f'<b style="color:#22c55e;">{len(sub)}</b>/{st.get("total","?")}' +
                f'<br/>{"  ".join(sub)}</div>'))
        except Exception:
            pass

r1_submit_btn.on_click(on_r1_submit)
display(widgets.HBox([r1_submit_btn, r1_status]))
display(r1_status_box)


<div style='background:linear-gradient(90deg,#1e1b4b,#0f172a);padding:20px 26px;border-radius:8px;border-left:4px solid #f59e0b;margin:20px 0 12px 0;font-family:Georgia,serif;'>
<div style='display:flex;align-items:baseline;gap:12px;margin-bottom:8px;'>
  <h2 style='color:#f59e0b;margin:0;'>Round 2 &#8212; The Images</h2>
  <span style='color:#475569;font-size:0.85em;'>Step 2 of 3</span>
</div>
<p style='color:#e2e8f0;margin:0 0 12px 0;font-size:0.95em;line-height:1.7;'>
  Each tab below shows infographics retained from Tesla&#8217;s annual report for the fiscal year preceding each earnings call, alongside the GAAP financial and KPI summary for that year. Use this visual and financial evidence to classify each call&#8217;s signal.
</p>
<div style='background:rgba(0,0,0,0.3);border-radius:6px;padding:12px 16px;margin-bottom:10px;'>
<p style='color:#f59e0b;font-size:0.82em;font-weight:bold;text-transform:uppercase;letter-spacing:0.05em;margin:0 0 8px 0;'>What to do in each tab</p>
<ul style='color:#94a3b8;font-size:0.88em;line-height:2;margin:0;padding-left:18px;'>
  <li>Look at the images: are they showing <strong style='color:#e2e8f0;'>positive KPIs</strong> (growth, scale, milestones) or <strong style='color:#e2e8f0;'>cautionary signals</strong> (comparisons, risk disclosures)?</li>
  <li>Check the <strong style='color:#e2e8f0;'>GAAP table</strong>: are revenue and margins growing? Is net income positive?</li>
  <li>Compare to what Musk <em>said</em> in Round 1. Do the numbers back him up, or contradict him?</li>
  <li>Then select your <strong style='color:#e2e8f0;'>Signal classification</strong>:
    <ul style='margin-top:4px;line-height:2;'>
      <li><strong>Consistent</strong> &#8212; text, images, and numbers all point the same direction</li>
      <li><strong>Divergent</strong> &#8212; at least one source contradicts the others</li>
      <li><strong>Unclear</strong> &#8212; insufficient information to call it either way</li>
    </ul>
  </li>
</ul>
</div>
<div style='background:rgba(245,158,11,0.08);border:1px solid rgba(245,158,11,0.3);border-radius:6px;padding:12px 16px;'>
<p style='color:#f59e0b;font-size:0.82em;font-weight:bold;text-transform:uppercase;letter-spacing:0.05em;margin:0 0 6px 0;'>&#9889; Power Play decision</p>
<p style='color:#94a3b8;font-size:0.88em;line-height:1.7;margin:0;'>After reviewing all four tabs, decide whether to declare a Power Play on one call. This is the call you feel <em>most confident</em> about after seeing the images. Your chips on that call will count &times;2 in the final score. This choice is optional, one-time, and irreversible &#8212; lock it in before submitting Round 2.</p>
</div>
</div>

In [ ]:
# @title 🟡 Round 2 — Annual report infographics {display-mode: "form"}
r2_banners = {}

def make_r2_tab(c):
    q, fy = c['quarter'], c['10k_fy']
    # R1 prediction reminder
    _r1p = STATE.get('r1_predictions', {}).get(q, None)
    _r1c = STATE['chips'].get(q, 3)
    if _r1p:
        _pc = '#22c55e' if 'rose' in _r1p.lower() else '#ef4444'
        _r1_banner = widgets.HTML(value=(
            f'<div style="background:#0f172a;border-left:3px solid {_pc};'
            f'padding:8px 12px;margin:0 0 10px 0;border-radius:0 4px 4px 0;'
            f'font-family:Georgia,serif;font-size:0.84em;">'
            f'<span style="color:#475569;">Your Round 1 call: </span>'
            f'<b style="color:{_pc};">{_r1p}</b>'
            f'<span style="color:#475569;"> — {_r1c} chips</span></div>'))
    else:
        _r1_banner = widgets.HTML(value=(
            '<div style="background:#0f172a;border-left:3px solid #334155;'
            'padding:8px 12px;margin:0 0 10px 0;border-radius:0 4px 4px 0;'
            'font-family:Georgia,serif;font-size:0.84em;color:#475569;">'
            'Submit Round 1 to see your prediction here.</div>'))
    r2_banners[q] = _r1_banner  # store for live update
    # Only show the primary (non-supplemented) images for this call's year
    primary_imgs = IMG_MFST[q][IMG_MFST[q]['imputed'] == False].reset_index(drop=True)
    GITHUB_IMG = ('https://raw.githubusercontent.com/iamjamesbowden/AG952/'
                  'main/week10/images/TSLA_{yr}/img_001.png')
    gallery = []
    if len(primary_imgs) == 0:
        gallery.append(widgets.HTML(value=(
            '<div style="background:#0f172a;border:1px dashed #334155;'
            'border-radius:6px;padding:24px;text-align:center;'
            'color:#475569;font-size:0.84em;margin:8px 0;">'
            'No infographic available for this call.'
            '</div>')))
    else:
        for _, row in primary_imgs.iterrows():
            src_fy = int(row.get('source_fy', fy))
            img_url = GITHUB_IMG.format(yr=src_fy)
            item = []
            try:
                req = urllib.request.Request(img_url,
                    headers={'User-Agent': 'Mozilla/5.0'})
                img_bytes = urllib.request.urlopen(req, timeout=10).read()
                item.insert(0, widgets.Image(value=img_bytes, format='png',
                    layout=widgets.Layout(max_width='600px', max_height='400px')))
            except Exception:
                item.insert(0, widgets.HTML(value=(
                    '<div style="background:#0f172a;border:1px dashed #334155;'
                    'border-radius:4px;padding:20px 12px;text-align:center;'
                    'color:#475569;font-size:0.75em;width:260px;">'
                    ' Annual Report infographic<br/>'
                    '<span style="font-size:0.85em;">Image loading…</span></div>')))
            gallery.append(widgets.VBox(item, layout=widgets.Layout(margin='4px')))

    gallery_box = widgets.HBox(gallery, layout=widgets.Layout(flex_flow='row wrap'))
    div_html = (
        f'<div style="font-family:Georgia,serif;">'
        f'<div style="color:#818cf8;font-size:0.9em;margin-bottom:8px;">'
        'What Tesla reported vs what Tesla showed</div>'
        f'<div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;">'
        f'<div>{build_gaap_table(fy)}</div>'
        f'<div>{build_kpi_table(fy)}</div>'
        f'</div></div>'
    )
    cls_w = widgets.RadioButtons(
        options=[
            '\U0001f4c8 Bullish \u2014 infographic and numbers suggest a positive outcome',
            '\U0001f4c9 Bearish \u2014 evidence suggests caution or underperformance',
            '\u2194 Mixed \u2014 signals are contradictory or inconclusive'],
        description='',
        layout=widgets.Layout(width='98%'))

    return widgets.VBox([
        _r1_banner,
        widgets.HTML(value='<div class="ag-label" style="margin:6px 0 4px 0;">Annual Report infographic</div>'),
        gallery_box,
        widgets.HTML(value='<hr style="border-color:#334155;margin:10px 0;">'),
        widgets.HTML(value=div_html),
        widgets.HTML(value=(
            '<div style="background:#1e293b;border-left:3px solid #818cf8;'
            'padding:10px 14px;margin:12px 0 4px 0;border-radius:0 6px 6px 0;">'
            '<div style="color:#818cf8;font-size:0.88em;font-weight:bold;margin-bottom:4px;">'
            'What does this evidence tell you?</div>'
            '<div style="color:#94a3b8;font-size:0.81em;line-height:1.5;">'
            'Consider the infographic, GAAP numbers, and non-financial KPIs together. '
            'What direction do they point for the market\u2019s reaction?</div></div>')),
        cls_w,
    ], layout=widgets.Layout(padding='10px')), cls_w

r2_tab      = widgets.Tab()
r2_classify = {}
_tab_contents = []
for _call in CALLS:
    _content, _cls = make_r2_tab(_call)
    _tab_contents.append(_content)
    r2_classify[_call['quarter']] = _cls
r2_tab.children = _tab_contents
for _i, _call in enumerate(CALLS):
    r2_tab.set_title(_i, ANON_MAP.get(_call['quarter'], _call['quarter']))
display(r2_tab)


In [ ]:
# @title 🟡 Round 2 — Chip adjustment & submit {display-mode: "form"}
def _make_chip_adj_widget(label_prefix='Round 1'):
    """Returns (vbox, get_deltas, set_disabled) for a zero-sum chip adjustment block."""
    _deltas   = {q: 0 for q in QUARTERS}
    _rows_box = []
    _btn_minus = {}
    _btn_plus  = {}
    _delta_lbl = {}
    _new_lbl   = {}
    _net_html  = widgets.HTML()

    def _refresh_net():
        net   = sum(_deltas.values())
        valid = all(1 <= STATE['chips'].get(q,3)+_deltas[q] <= 4 for q in QUARTERS)
        col   = '#22c55e' if net == 0 and valid else '#ef4444'
        if net == 0 and valid:
            note = u' ✓ balanced — ready to submit'
        elif net != 0:
            direction = 'Subtract' if net > 0 else 'Add'
            amount    = abs(net)
            note = f' — net {net:+d}: {direction} {amount} chip{"s" if amount>1 else ""} on another call'
        else:
            note = ' — a call is out of range (min 1, max 4)'
        _net_html.value = (
            f'<span style="color:{col};font-family:Georgia,serif;font-size:0.86em;">'
            f'Net change: {net:+d}{note}</span>')

    def _mk_minus(q_):
        def _fn(b):
            if _deltas[q_] > -1: 
                _deltas[q_] -= 1
                _update_row(q_)
                _refresh_net()
        return _fn

    def _mk_plus(q_):
        def _fn(b):
            if _deltas[q_] < 1:
                _deltas[q_] += 1
                _update_row(q_)
                _refresh_net()
        return _fn

    def _update_row(q_):
        d   = _deltas[q_]
        new = STATE['chips'].get(q_, 3) + d
        d_col = '#22c55e' if d > 0 else '#ef4444' if d < 0 else '#94a3b8'
        d_str = f'+{d}' if d > 0 else (str(d) if d != 0 else u'±0')
        _delta_lbl[q_].value = (
            f'<span style="color:{d_col};font-weight:bold;font-size:1em;'
            f'font-family:Georgia,serif;min-width:28px;display:inline-block;'
            f'text-align:center;">{d_str}</span>')
        n_col = '#ef4444' if (new < 1 or new > 4) else '#e2e8f0'
        _new_lbl[q_].value = (
            f'<span style="color:{n_col};font-weight:bold;font-family:Georgia,serif;'
            f'font-size:0.92em;">→ {new} chip{"s" if new!=1 else ""}</span>')

    for _call in CALLS:
        q_    = _call['quarter']
        anon  = ANON_MAP.get(q_, q_)
        base  = STATE['chips'].get(q_, 3)
        _btn_minus[q_] = widgets.Button(
            description='−1', button_style='',
            layout=widgets.Layout(width='44px', height='32px'))
        _btn_plus[q_]  = widgets.Button(
            description='+1', button_style='',
            layout=widgets.Layout(width='44px', height='32px'))
        _delta_lbl[q_] = widgets.HTML()
        _new_lbl[q_]   = widgets.HTML()
        _call_lbl      = widgets.HTML(
            value=f'<span style="color:#e2e8f0;font-family:Georgia,serif;'
                  f'font-size:0.9em;min-width:80px;display:inline-block;">'
                  f'{anon}</span>'
                  f'<span style="color:#94a3b8;font-family:Georgia,serif;'
                  f'font-size:0.82em;margin-left:4px;">({base} chips)</span>',
            layout=widgets.Layout(min_width='160px'))
        _btn_minus[q_].on_click(_mk_minus(q_))
        _btn_plus[q_].on_click(_mk_plus(q_))
        _update_row(q_)
        _rows_box.append(widgets.HBox(
            [_call_lbl, _btn_minus[q_], _delta_lbl[q_], _btn_plus[q_], _new_lbl[q_]],
            layout=widgets.Layout(align_items='center', margin='3px 0')))

    def get_deltas():
        return dict(_deltas)

    def set_disabled(disabled):
        for q_ in QUARTERS:
            _btn_minus[q_].disabled = disabled
            _btn_plus[q_].disabled  = disabled

    _refresh_net()
    container = widgets.VBox(_rows_box + [_net_html],
        layout=widgets.Layout(margin='8px 0 12px 0'))
    return container, get_deltas, set_disabled

display(HTML('<div class="ag-section-hdr">Round 2 &#8212; Chip adjustment &amp; submission</div>'))

display(HTML('<div style="background:#1e293b;border:1px solid #334155;'
             'border-radius:8px;padding:14px 18px;margin:14px 0 8px 0;font-family:Georgia,serif;">'
             '<div style="color:#e2e8f0;font-size:0.92em;font-weight:bold;margin-bottom:6px;">'
             ' Optional: shift chips between calls</div>'
             '<div style="color:#94a3b8;font-size:0.82em;line-height:1.6;">'
             'Use <b style="color:#e2e8f0;">−1</b> and <b style="color:#e2e8f0;">+1</b> '
             'to move chips. What you add to one call must come from another — '
             'the net change must be <b>zero</b> to submit. '
             'Min 1 chip per call, max 4.'
             '</div></div>'))

_r2_adj_box, _r2_get_deltas, _r2_set_disabled = _make_chip_adj_widget('Round 1')
display(_r2_adj_box)

r2_submit_btn = widgets.Button(description='Submit Round 2', button_style='success',
    layout=widgets.Layout(width='200px', height='38px'))
r2_status     = widgets.HTML()
r2_status_box = widgets.Output()

def on_r2_submit(b):
    if not STATE.get('r1_submitted'):
        r2_status.value = '<span style="color:#ef4444;">Submit Round 1 first.</span>'
        return
    if STATE.get('r2_submitted'):
        r2_status.value = '<span style="color:#f59e0b;">Already submitted.</span>'
        return
    _deltas = _r2_get_deltas()
    _net = sum(_deltas.values())
    if _net != 0:
        r2_status.value = (f'<span style="color:#ef4444;">Net chip change must be zero '
                           f'(currently {_net:+d}). Balance before submitting.</span>')
        return
    for _q in QUARTERS:
        _new = STATE['chips'].get(_q, 3) + _deltas[_q]
        if _new < 1 or _new > 4:
            r2_status.value = (f'<span style="color:#ef4444;">'
                               f'{ANON_MAP.get(_q,_q)} would have {_new} chips — must be 1–4.</span>')
            return
    for _q in QUARTERS:
        STATE['chips'][_q] = STATE['chips'].get(_q, 3) + _deltas[_q]
    payload = {'action': 'submit_round2', 'team_name': STATE['team_name'],
               'calls': [{'quarter': q, 'classification': r2_classify[q].value}
                          for q in QUARTERS]}
    try:
        resp = post_to_script(payload)
        if not resp.get('success'):
            r2_status.value = f'<span style="color:#ef4444;">{resp.get("message")}</span>'
            return
    except Exception:
        pass
    STATE['r2_submitted'] = True
    _r2_set_disabled(True)
    r2_submit_btn.disabled = True
    for q in QUARTERS:
        r2_classify[q].disabled = True
    try:
        _r3_refresh_chips()
    except NameError:
        pass
    r2_status.value = ('<span style="color:#22c55e;">&#10003; Round 2 submitted. '
                       'Explore the Social Media Pulse section below while you wait, '
                       'then continue to Round 3 when the facilitator unlocks it.</span>')
    with r2_status_box:
        try:
            st  = get_status(2)
            sub = st.get('submitted_teams', [])
            display(HTML(f'<div class="ag-status-box">Round 2 submissions: '
                f'<b style="color:#22c55e;">{len(sub)}</b>/{st.get("total","?")}'
                f'<br/>{ " &nbsp; ".join(sub)}</div>'))
        except Exception:
            pass

r2_submit_btn.on_click(on_r2_submit)
display(widgets.HBox([r2_submit_btn, r2_status]))
display(r2_status_box)


In [ ]:
# @title 📱 Social Media Pulse — Twitter &amp; #Tesla {display-mode: "form"}
# Real tweets mentioning #tesla scraped on 11 Jul 2022, nine days before the
# Q2 2022 earnings call. This is NOT one of your four prediction calls.
# Work through the analysis, then make your hypothesis below.

import re
from collections import Counter

_tweets_path = dpath('tesla_tweets_en.csv')  # in clips/ same as DRIVE_ROOT
try:
    TWEETS = pd.read_csv(_tweets_path)
    # ── Offensive-content filter ─────────────────────────────────────────────
    # Two-stage: (1) remove offensive tweets entirely, (2) strip any offensive
    # words that survive from the word-frequency word list.
    # Patterns use regex so obfuscated variants (n*gga, nigg@, r3tard…) are caught.
    import re as _re
    _OFFENSIVE_PATS = [
        # N-word — all common spellings and obfuscations
        r'n[i1!|]gg[a@4e]', r'nigg', r'n[\*\-\_\.][a-z]*gg',
        # Other racial / ethnic slurs
        r'ch[i1]nk', r'sp[i1][ck]', r'k[i1]ke', r'g[o0]{2}k',
        r'w[e3]tb[a@]ck', r'c[o0]{2}n',
        # Homophobic slurs
        r'fagg?[o0]t', r'dyke',
        # Ableist slurs
        r'r[e3]t[a@]rd',
        # Misogynistic slurs
        r'c[u\*]nt', r'wh[o0]r[e3]', r'sl[u\*]t',
    ]
    def _compile_pats():
        return [_re.compile(p, _re.IGNORECASE) for p in _OFFENSIVE_PATS]
    _COMPILED = _compile_pats()

    def _normalise(text):
        """Strip punctuation/symbols that may be used to obfuscate slurs."""
        return _re.sub(r'[^a-z0-9\s]', '', str(text).lower())

    def _is_offensive(text):
        raw  = str(text).lower()
        norm = _normalise(text)
        return any(p.search(raw) or p.search(norm) for p in _COMPILED)

    _before = len(TWEETS)
    TWEETS = TWEETS[~TWEETS['tweet'].apply(_is_offensive)].reset_index(drop=True)
    _n = len(TWEETS)
    _loaded = True
except Exception as _e:
    display(HTML(f'<div class="ag-status-box" style="color:#ef4444;">'
                 f'Tweet data not found at {_tweets_path}: {_e}'
                 f'<br/>Upload tesla_tweets_en.csv to workshop_data/ in Drive.</div>'))
    TWEETS = pd.DataFrame(columns=['tweet','nlikes','nretweets','nreplies'])
    _n = 0; _loaded = False

display(HTML(f'''
<div style="background:#1e293b;border:1px solid #334155;border-radius:8px;
            padding:16px;margin:8px 0;font-family:Georgia,serif;">
  <div style="color:#818cf8;font-size:1.05em;font-weight:bold;margin-bottom:8px;">
    📱 Social Media Pulse: Twitter &amp; #Tesla
  </div>
  <div style="color:#94a3b8;font-size:0.88em;line-height:1.6;">
    <b style="color:#e2e8f0;">{_n:,} English tweets</b> mentioning <code>#tesla</code>,
    scraped on <b style="color:#fbbf24;">11 July 2022</b> &#8212; nine days before the
    Q2 2022 earnings call (not one of your four prediction calls).<br/>
    Work through the NLP analysis below, then make your hypothesis.
  </div>
</div>'''))

# ── Preprocessing ─────────────────────────────────────────────────────────────
TWITTER_STOPS = {
    'tesla','the','a','an','is','are','was','were','be','been','have','has','had',
    'do','does','did','will','would','could','should','may','might','shall',
    'and','or','but','for','not','with','this','that','it','its','of','in','on',
    'to','at','by','from','as','up','out','if','no','so','than','then','when',
    'there','their','they','we','you','he','she','i','me','my','your','our',
    'his','her','who','which','what','about','into','after','more','also',
    'just','all','can','get','got','go','one','new','like','amp','rt',
    'https','http','co','t','s','im','don','re','ve','ll','us','its',
}

def _clean_tw(text):
    text = re.sub(r'http\S+', '', str(text))
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    return text.lower()

_all_words = []
for _txt in TWEETS['tweet']:
    _all_words.extend(
        w for w in _clean_tw(_txt).split()
        if len(w) > 2 and w not in TWITTER_STOPS and not _is_offensive(w))
_top_words = Counter(_all_words).most_common(20)

# ── Charts ────────────────────────────────────────────────────────────────────
_fig, _axes = plt.subplots(1, 2, figsize=(13, 4), facecolor='#1e293b')

_ax0 = _axes[0]
if _top_words:
    _words, _counts = zip(*_top_words)
    _ax0.barh(range(len(_words)), _counts, color='#3b82f6', edgecolor='none')
    _ax0.set_yticks(range(len(_words)))
    _ax0.set_yticklabels(_words, fontsize=8, color='#e2e8f0')
    _ax0.invert_yaxis()
_ax0.set_xlabel('Frequency', color='#94a3b8', fontsize=8)
_ax0.set_title('Top 20 terms in #Tesla tweets\n(11 Jul 2022, English, originals only)',
               color='#e2e8f0', fontsize=9, pad=6)
_ax0.set_facecolor('#0f172a')
_ax0.tick_params(axis='x', colors='#94a3b8', labelsize=7)
for _sp in ['top','right']: _ax0.spines[_sp].set_visible(False)
for _sp in ['bottom','left']: _ax0.spines[_sp].set_color('#334155')

_ax1 = _axes[1]
_sentiment_summary = 'VADER not installed (pip install vaderSentiment)'
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer as _VADER
    _va = _VADER()
    _sc = TWEETS['tweet'].apply(lambda t: _va.polarity_scores(str(t))['compound'])
    _pos = int((_sc >= 0.05).sum()); _neg = int((_sc <= -0.05).sum())
    _neu = len(_sc) - _pos - _neg
    _ax1.bar(['Positive','Neutral','Negative'], [_pos, _neu, _neg],
             color=['#22c55e','#9ca3af','#ef4444'], width=0.5)
    _ax1.set_facecolor('#0f172a')
    _ax1.tick_params(colors='#94a3b8', labelsize=8)
    for _sp in ['top','right']: _ax1.spines[_sp].set_visible(False)
    for _sp in ['bottom','left']: _ax1.spines[_sp].set_color('#334155')
    _ax1.set_ylabel('Tweet count', color='#94a3b8', fontsize=8)
    _ax1.set_title('VADER sentiment distribution', color='#e2e8f0', fontsize=9, pad=6)
    _sentiment_summary = (f'{100*_pos/len(_sc):.0f}% positive, '
                          f'{100*_neg/len(_sc):.0f}% negative, '
                          f'{100*_neu/len(_sc):.0f}% neutral')
except ImportError:
    _ax1.text(0.5,0.5,'Run: pip install vaderSentiment\nthen re-run this cell',
              ha='center',va='center',color='#94a3b8',fontsize=9,
              transform=_ax1.transAxes)
    _ax1.set_facecolor('#0f172a')

plt.suptitle('Twitter #Tesla text analysis &#8212; 11 July 2022',
             color='#818cf8', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

# ── Top tweets ────────────────────────────────────────────────────────────────
display(HTML('<div class="ag-label" style="margin:14px 0 6px 0;">'             ' Top 10 most-liked #Tesla tweets (11 Jul 2022)</div>'))
_top5 = TWEETS.nlargest(10, 'nlikes')[['tweet','nlikes','nretweets']].reset_index(drop=True)
_cards_html = ''
for _, _row in _top5.iterrows():
    _t = str(_row['tweet'])
    if _is_offensive(_t):   # final safety check before display
        continue
    _t_disp = (_t[:260] + '…') if len(_t) > 260 else _t
    _cards_html += (
        f'<div style="background:#0f172a;border:1px solid #334155;border-radius:6px;'        f'padding:10px 14px;margin:5px 0;font-size:0.84em;color:#e2e8f0;">'        f'{_t_disp}<br/>'        f'<span style="color:#94a3b8;font-size:0.78em;">'        f'&hearts; {int(_row["nlikes"]):,} &nbsp; 🔁 {int(_row["nretweets"]):,}</span></div>')
display(HTML(f'<div style="font-family:Georgia,serif;">{_cards_html}</div>'))

# ── Hypothesis widget ─────────────────────────────────────────────────────────
display(HTML('''<div style="background:#1e293b;border:1px solid #818cf8;border-radius:8px;
padding:14px 16px;margin:16px 0 8px 0;font-family:Georgia,serif;">
  <div style="color:#818cf8;font-weight:bold;margin-bottom:6px;">
     Your hypothesis: does Twitter predict the market?
  </div>
  <div style="color:#94a3b8;font-size:0.88em;line-height:1.5;">
    Based on the word frequency and sentiment above, do you think the Q2 2022
    earnings call (20 Jul 2022) produced a <b style="color:#e2e8f0;">positive or negative
    5-day abnormal return</b> relative to the S&amp;P 500?
  </div>
</div>'''))

tw_vote = widgets.RadioButtons(
    options=['📈 Positive — market response will be positive (stock outperforms S&P 500)',
             '📉 Negative — market response will be negative (stock underperforms S&P 500)',
             '🤷 Uncertain — social media is too noisy to call'],
    layout=widgets.Layout(width='98%'))

display(HTML('''<div style="background:#1e293b;border-left:3px solid #f59e0b;
padding:10px 14px;margin:14px 0 6px 0;border-radius:0 6px 6px 0;font-family:Georgia,serif;">
  <div style="color:#f59e0b;font-size:0.88em;font-weight:bold;margin-bottom:4px;">
     Bonus chip bet (optional — side bet, does not affect your four scored calls)
  </div>
  <div style="color:#94a3b8;font-size:0.82em;line-height:1.5;">
    Stake 0–3 bonus chips on your hypothesis. Correct = +chips bonus; wrong = −chips penalty.
    This is tracked locally and does <em>not</em> affect your earnings-call predictions.
  </div>
</div>'''))

tw_bet_w = widgets.IntSlider(
    value=1, min=0, max=3, step=1,
    description='Bonus chips:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='340px'))
tw_bet_preview = widgets.HTML()

def _update_tw_preview(change=None):
    n = tw_bet_w.value
    if n == 0:
        tw_bet_preview.value = '<span style="color:#475569;font-size:0.82em;">No bet — watching only.</span>'
    else:
        tw_bet_preview.value = (
            f'<span style="color:#94a3b8;font-size:0.82em;">'
            f'Correct: <b style="color:#22c55e;">+{n} chips</b>'
            f' &nbsp;&nbsp; Wrong: <b style="color:#ef4444;">−{n} chips</b></span>')

tw_bet_w.observe(_update_tw_preview, names='value')
_update_tw_preview()

tw_reveal_btn = widgets.Button(description='Reveal Q2 2022 outcome',
    button_style='warning', layout=widgets.Layout(width='220px'))
tw_reveal_out = widgets.Output()

def _on_tw_reveal(b):
    tw_reveal_btn.disabled = True
    _actual_negative = True  # Q2 2022 = −9.4% abnormal return
    _user_neg = '📉' in tw_vote.value
    _user_pos = '📈' in tw_vote.value
    _bet = tw_bet_w.value
    if _bet > 0:
        _won = (_actual_negative and _user_neg) or (not _actual_negative and _user_pos)
        STATE['twitter_score'] = _bet if _won else -_bet
        STATE['twitter_chips'] = _bet
        _bet_result = (f'<b style="color:#22c55e;">+{_bet} chips — correct!</b>' if _won
                       else f'<b style="color:#ef4444;">−{_bet} chips — wrong</b>')
    else:
        STATE['twitter_score'] = 0
        STATE['twitter_chips'] = 0
        _bet_result = '<span style="color:#475569;">No bet placed.</span>'
    with tw_reveal_out:
        display(HTML(f'''
        <div style="background:#0f172a;border:2px solid #ef4444;border-radius:8px;
             padding:16px;margin:8px 0;font-family:Georgia,serif;">
          <div style="color:#ef4444;font-size:1.05em;font-weight:bold;margin-bottom:8px;">
            📉 Actual Q2 2022 result: −9.4% abnormal return
          </div>
          <div style="color:#e2e8f0;font-size:0.86em;line-height:1.6;">
            The Q2 2022 earnings call (20 Jul 2022) produced a
            <b style="color:#ef4444;">−9.4% five-day abnormal return</b> relative to the S&amp;P 500.
            <b>Bonus bet: {_bet_result}</b><br/><br/>
            Despite <b>{_sentiment_summary}</b> in the Twitter conversation nine days before
            the call, Tesla sharply underperformed expectations afterwards.<br/><br/>
            <b style="color:#fbbf24;"> Why does Twitter struggle to predict this?</b>
            These tweets were written <em>before</em> the call — they capture retail
            expectations and hype, not what was actually communicated on the call itself.
            The market reaction is driven by whether the call met, beat, or missed those
            expectations, and by signals embedded in the language: how management frames
            guidance, where hesitation appears in the transcript, how vocal features shift
            under pressure. That is what transcript-level text analytics is designed to
            surface — and what a pre-call tweet stream cannot tell you (Todd et al., 2025).
          </div>
        </div>'''))

tw_reveal_btn.on_click(_on_tw_reveal)
display(tw_vote)
display(widgets.VBox([tw_bet_w, tw_bet_preview]))
display(widgets.HBox([tw_reveal_btn]))
display(tw_reveal_out)
display(HTML('<div style="color:#475569;font-size:0.76em;margin-top:6px;font-family:Georgia,serif;">'
             'Dataset: English-language original tweets mentioning #tesla, '
             'scraped 11 Jul 2022. For educational use only.</div>'))


<div style='background:linear-gradient(90deg,#1e1b4b,#0f172a);padding:20px 26px;border-radius:8px;border-left:4px solid #3b82f6;margin:20px 0 12px 0;font-family:Georgia,serif;'>
<div style='display:flex;align-items:baseline;gap:12px;margin-bottom:8px;'>
  <h2 style='color:#3b82f6;margin:0;'>Round 3 &#8212; The Audio</h2>
  <span style='color:#475569;font-size:0.85em;'>Step 3 of 3</span>
</div>
<p style='color:#e2e8f0;margin:0 0 12px 0;font-size:0.95em;line-height:1.7;'>
  Paralinguistic features have been extracted from Elon Musk&#8217;s voice across every 5-minute window of each earnings call Q&amp;A. For each call you see two visualisations. Review them, then update (or confirm) your rationale before the final submission.
</p>
<div style='display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:12px;'>
  <div style='background:rgba(0,0,0,0.3);border-radius:6px;padding:12px 14px;'>
    <p style='color:#3b82f6;font-size:0.82em;font-weight:bold;text-transform:uppercase;letter-spacing:0.05em;margin:0 0 8px 0;'>Silence map</p>
    <ul style='color:#94a3b8;font-size:0.85em;line-height:1.9;margin:0;padding-left:16px;'>
      <li>Each bar = one 5-minute window of the Q&amp;A</li>
      <li><span style='color:#f59e0b;'>&#9632; Amber</span> = more silence than average; <span style='color:#3b82f6;'>&#9632; Blue</span> = less silence</li>
      <li>Long pauses can signal hesitation, careful framing, or stress</li>
      <li>Sustained amber in the later windows may indicate Musk was under pressure from analysts</li>
      <li>The purple dashed line marks the &#8220;key question&#8221; window with the highest sentiment divergence</li>
    </ul>
  </div>
  <div style='background:rgba(0,0,0,0.3);border-radius:6px;padding:12px 14px;'>
    <p style='color:#3b82f6;font-size:0.82em;font-weight:bold;text-transform:uppercase;letter-spacing:0.05em;margin:0 0 8px 0;'>Feature radar</p>
    <ul style='color:#94a3b8;font-size:0.85em;line-height:1.9;margin:0;padding-left:16px;'>
      <li><span style='color:#60a5fa;'>&#9632; Blue polygon</span> = this call; <span style='color:#fbbf24;'>&#9632; Amber dashed</span> = cross-call average</li>
      <li>Each axis is a vocal feature z-scored against the full dataset (0.5 = average)</li>
      <li>&#9679; Amber dot = feature is more than 1 SD from the average &#8212; noteworthy deviation</li>
      <li>High pitch + high intensity = animated, emphatic delivery (often bullish tone)</li>
      <li>High silence + high voice breaks = more hesitant, less fluent speech</li>
      <li>Click &#8220;What do the radar axes mean?&#8221; below each chart for full definitions</li>
    </ul>
  </div>
</div>
<div style='background:rgba(59,130,246,0.08);border:1px solid rgba(59,130,246,0.3);border-radius:6px;padding:12px 16px;'>
<p style='color:#3b82f6;font-size:0.82em;font-weight:bold;text-transform:uppercase;letter-spacing:0.05em;margin:0 0 6px 0;'>Then for each call</p>
<p style='color:#94a3b8;font-size:0.88em;line-height:1.7;margin:0;'>Update or confirm your prediction and write a new rationale (max 30 words) that references at least one feature from the audio. Does the vocal evidence support or undermine what you concluded from the text and images? Submit when done and wait for The Reveal.</p>
</div>
</div>

In [ ]:
# @title 🟢 Round 3 — Audio analysis {display-mode: "form"}
r3_reason_areas = {}
r3_word_cntrs   = {}

for _call in CALLS:
    _q = _call['quarter']
    display(HTML(f'<div class="ag-section-hdr" style="border-left-color:#3b82f6;">'
                 f'{ANON_MAP.get(_q, _q)}</div>'))
    make_silence_map(_call).show()
    make_radar(_call).show()
    r3_reason_areas[_q] = widgets.Textarea(
        placeholder=f'Update or confirm your Round 1 rationale for {ANON_MAP.get(_q,_q)} (max 30 words) …',
        layout=widgets.Layout(width='720px', height='64px'))
    r3_word_cntrs[_q] = widgets.HTML(value='<div class="ag-word-counter">0 / 30 words</div>')
    def _mk_wc3(_q_=_q):
        def _upd(ch):
            wc = len(ch['new'].split())
            col = '#ef4444' if wc > 30 else '#94a3b8'
            r3_word_cntrs[_q_].value = (f'<div class="ag-word-counter" '
                f'style="color:{col};">{wc} / 30 words</div>')
        return _upd
    r3_reason_areas[_q].observe(_mk_wc3(), names='value')
    display(widgets.HTML(f'<span style="color:#94a3b8;font-size:0.85em;">'
                          f'Update reasoning for {ANON_MAP.get(_q,_q)}:</span>'))
    display(widgets.VBox([r3_reason_areas[_q], r3_word_cntrs[_q]]))
    display(HTML('<hr style="border-color:#1e293b;margin:12px 0;">'))

In [ ]:
# @title 🟢 Round 3 — Final submission {display-mode: "form"}
display(HTML('<div class="ag-section-hdr" style="border-left-color:#3b82f6;">'
             'Round 3 — Final submission</div>'))

# ── Audio-informed chip adjustment ───────────────────────────────────────────
display(HTML('<div style="background:#1e293b;border:1px solid #334155;border-radius:8px;'
             'padding:14px 18px;margin:10px 0;font-family:Georgia,serif;">'
             '<div style="color:#e2e8f0;font-size:0.92em;font-weight:bold;margin-bottom:6px;">'
             ' Optional: shift chips based on audio evidence</div>'
             '<div style="color:#94a3b8;font-size:0.82em;line-height:1.6;">'
             'Use <b style="color:#e2e8f0;">−1</b> and <b style="color:#e2e8f0;">+1</b> '
             'to move chips. Net change must be <b>zero</b>. '
             'Min 1 chip per call, max 4. Leave unchanged to keep Round 2 allocation.'
             '</div></div>'))

_r3_chip_out = widgets.Output()

def _r3_refresh_chips():
    with _r3_chip_out:
        clear_output(wait=True)
        rows = ''
        for q in QUARTERS:
            anon  = ANON_MAP.get(q, q)
            chips = STATE['chips'].get(q, 3)
            rows += (f'<td style="padding:4px 10px;color:#94a3b8;">{anon}</td>'
                     f'<td style="padding:4px 8px;color:#f59e0b;font-weight:bold;'
                     f'font-size:1.05em;">{chips}</td>')
        display(HTML(
            f'<div style="font-family:Georgia,serif;font-size:0.84em;'
            f'color:#94a3b8;margin:4px 0 2px 0;">Current allocation after Round 2:</div>'
            f'<table style="border-collapse:collapse;margin:2px 0 8px 0;">'
            f'<tr>{rows}</tr></table>'))

_r3_refresh_chips()
display(_r3_chip_out)

_r3_adj_box, _r3_get_deltas, _r3_set_disabled = _make_chip_adj_widget('Round 2')

def _r3_update_adj_pp():
    try: _r3_update_pp_preview()
    except NameError: pass

display(_r3_adj_box)

# ── Power Play ────────────────────────────────────────────────────────────────
display(HTML('''<div class="pp-box">
  <div style="color:#f59e0b;font-size:1em;font-weight:bold;margin-bottom:8px;">
    ⚡ Power Play — optional, one-time, irreversible
  </div>
  <div style="color:#94a3b8;font-size:0.86em;line-height:1.6;">
    Now you have seen <em>all</em> the evidence — text, infographics, and audio — pick the
    one call you are most confident about.
    Your chips on that call count <b style="color:#f59e0b;">×2</b> in the final score.<br/>
    <span style="color:#475569;font-size:0.9em;">Leave as "No Power Play" to skip.</span>
  </div>
</div>'''))

_r3_pp_options = ['No Power Play'] + ANON_QUARTERS
r3_pp_dropdown = widgets.Dropdown(
    options=_r3_pp_options, value='No Power Play',
    description='Power Play:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='300px'))
r3_pp_stake    = widgets.HTML()
r3_pp_lock_btn = widgets.Button(description='Lock in Power Play', button_style='warning',
    layout=widgets.Layout(width='170px'))
r3_pp_status   = widgets.HTML()

def _r3_update_pp_preview(change=None):
    choice = r3_pp_dropdown.value
    if choice == 'No Power Play':
        r3_pp_stake.value = ('<div style="color:#475569;font-size:0.84em;'
                             'font-family:Georgia,serif;padding:6px 0;">'
                             'No Power Play: all calls score at &times;1.</div>')
        return
    q     = next((k for k,v in ANON_MAP.items() if v == choice), None)
    chips = STATE['chips'].get(q, 3)
    r3_pp_stake.value = (
        f'<div style="background:#0f172a;border-radius:6px;padding:10px 14px;'
        f'font-family:Georgia,serif;font-size:0.86em;margin:4px 0 8px 0;">'
        f'<span style="color:#94a3b8;">If correct: </span>'
        f'<span style="color:#22c55e;font-weight:bold;">+{chips*2} pts</span>'
        f'&nbsp;&nbsp;&nbsp;'
        f'<span style="color:#94a3b8;">If wrong: </span>'
        f'<span style="color:#ef4444;font-weight:bold;">&#8722;{chips*2} pts</span>'
        f'&nbsp;&nbsp;&nbsp;'
        f'<span style="color:#475569;font-size:0.9em;">({chips} chips &times;2)</span>'
        f'</div>')

r3_pp_dropdown.observe(_r3_update_pp_preview, names='value')
_r3_update_pp_preview()

def on_r3_pp_lock(b):
    if STATE.get('r3_submitted'):
        r3_pp_status.value = '<span style="color:#f59e0b;">Round 3 already submitted.</span>'
        return
    if STATE.get('_r3_pp_locked'):
        r3_pp_status.value = '<span style="color:#f59e0b;">Power Play already locked.</span>'
        return
    choice = r3_pp_dropdown.value
    if choice == 'No Power Play':
        STATE['power_play_quarter'] = None
        r3_pp_status.value = '<span style="color:#94a3b8;">No Power Play — all calls ×1.</span>'
    else:
        q = next((k for k,v in ANON_MAP.items() if v == choice), None)
        STATE['power_play_quarter'] = q
        chips = STATE['chips'].get(q, 3)
        r3_pp_status.value = (
            f'<span style="color:#f59e0b;">⚡ Power Play locked on {choice}! '
            f'{chips} chips ×2: +{chips*2} if right, −{chips*2} if wrong.</span>')
    STATE['_r3_pp_locked'] = True
    r3_pp_dropdown.disabled = True
    r3_pp_lock_btn.disabled = True

r3_pp_lock_btn.on_click(on_r3_pp_lock)
display(widgets.VBox([
    widgets.HBox([r3_pp_dropdown, r3_pp_lock_btn]),
    r3_pp_stake,
    r3_pp_status
]))

# ── Submit ────────────────────────────────────────────────────────────────────
r3_submit_btn = widgets.Button(description='Submit Final Predictions',
    button_style='success', layout=widgets.Layout(width='240px', height='38px'))
r3_status     = widgets.HTML()
r3_status_box = widgets.Output()

def on_r3_submit(b):
    if not STATE.get('r2_submitted'):
        r3_status.value = '<span style="color:#ef4444;">Submit Round 2 first.</span>'
        return
    if STATE.get('r3_submitted'):
        r3_status.value = '<span style="color:#f59e0b;">Already submitted.</span>'
        return
    _r3_deltas = _r3_get_deltas()
    _r3_net    = sum(_r3_deltas.values())
    if _r3_net != 0:
        r3_status.value = (f'<span style="color:#ef4444;">Net chip change must be zero '
                           f'(currently {_r3_net:+d}). Balance before submitting.</span>')
        return
    for _q in QUARTERS:
        _new = STATE['chips'].get(_q, 3) + _r3_deltas[_q]
        if _new < 1 or _new > 4:
            r3_status.value = (f'<span style="color:#ef4444;">'
                               f'{ANON_MAP.get(_q,_q)} would have {_new} chips — must be 1–4.</span>')
            return
    for _q in QUARTERS:
        STATE['chips'][_q] = STATE['chips'].get(_q, 3) + _r3_deltas[_q]
    if not STATE.get('_r3_pp_locked'):
        r3_status.value = ('<span style="color:#ef4444;">'
                           'Lock your Power Play (or choose No Power Play) before submitting.</span>')
        return
    for q in QUARTERS:
        if len(r3_reason_areas[q].value.split()) > 30:
            r3_status.value = (f'<span style="color:#ef4444;">'
                               f'{ANON_MAP.get(q,q)} rationale over 30 words.</span>')
            return
    _pp_q = STATE.get('power_play_quarter')
    payload = {'action': 'submit_round3', 'team_name': STATE['team_name'],
               'calls': [{'quarter': q, 'chips': STATE['chips'][q],
                           'prediction': pred_dropdowns[q].value,
                           'reasoning': (r3_reason_areas[q].value.strip()
                                          or reason_areas[q].value.strip()),
                           'power_play': 1 if q == _pp_q else 0}
                          for q in QUARTERS]}
    try:
        resp = post_to_script(payload)
        if not resp.get('success'):
            r3_status.value = f'<span style="color:#ef4444;">{resp.get("message")}</span>'
            return
    except Exception:
        pass
    STATE['r3_submitted'] = True
    for q in QUARTERS:
        r3_reason_areas[q].disabled = True
    _r3_set_disabled(True)
    r3_submit_btn.disabled = True
    r3_pp_lock_btn.disabled = True
    _pp_anon = ANON_MAP.get(_pp_q, '—') if _pp_q else 'none'
    r3_status.value = (f'<span style="color:#22c55e;">&#10003; Final predictions submitted'
                       + (f' — Power Play on {_pp_anon}' if _pp_q else '')
                       + '. Scroll down for The Reveal.</span>')
    with r3_status_box:
        try:
            st  = get_status(3)
            sub = st.get('submitted_teams', [])
            display(HTML(f'<div class="ag-status-box">Final submissions: '
                f'<b style="color:#22c55e;">{len(sub)}</b>/{st.get("total","?")}'
                f'<br/>{" &nbsp; ".join(sub)}</div>'))
        except Exception:
            pass

r3_submit_btn.on_click(on_r3_submit)
display(widgets.HBox([r3_submit_btn, r3_status]))
display(r3_status_box)


<div style='background:linear-gradient(90deg,#1e1b4b,#0f172a);padding:20px 26px;border-radius:8px;border-left:4px solid #22c55e;margin:20px 0 12px 0;font-family:Georgia,serif;'>
<h2 style='color:#22c55e;margin:0 0 8px 0;'>The Reveal</h2>
<p style='color:#e2e8f0;margin:0 0 8px 0;font-size:0.95em;line-height:1.7;'>
  The facilitator has unlocked the reveal. Scroll down to see the actual 5-day abnormal returns for each call, your score breakdown, and the class leaderboard on the projection screen.
</p>
<p style='color:#94a3b8;margin:0;font-size:0.88em;line-height:1.6;'>
  Scoring: <strong style='color:#e2e8f0;'>+chips</strong> for each correct direction, <strong style='color:#ef4444;'>&#8722;chips</strong> for each wrong direction. Power Play calls count &times;2. Your individual reflection follows at the end.
</p>
</div>

In [ ]:
# @title 🏆 Results — Score reveal {display-mode: "form"}
_qtrs   = [c['quarter'] for c in CALLS]
_anons  = [ANON_MAP.get(q, q) for q in _qtrs]
_rets   = [c['returns']['abnormal_ret_pct'] for c in CALLS]
_dirs   = [c['returns']['direction'] for c in CALLS]
_colors = [CLR['ret_positive'] if d=='positive' else CLR['ret_negative'] for d in _dirs]
_labels = [f"{a}<br>({q})<br>{r:+.1f}%" for a, q, r in zip(_anons, _qtrs, _rets)]

_frames = [go.Frame(
    data=[go.Bar(x=_anons[:i+1], y=_rets[:i+1], text=_labels[:i+1],
               textposition='outside', textfont=dict(color='#e2e8f0', size=11),
               marker_color=_colors[:i+1])],
    name=str(i)) for i in range(len(_qtrs))]

_fig_reveal = go.Figure(
    data=[go.Bar(x=[], y=[], marker_color=[])],
    frames=_frames,
    layout=go.Layout(
        title=dict(text='5-Day Abnormal Returns — Actual Outcomes',
                   font=dict(color='#e2e8f0', size=15)),
        xaxis=dict(color='#94a3b8', gridcolor='#1e293b'),
        yaxis=dict(title='Abnormal return (%)', color='#94a3b8',
                   gridcolor='#1e293b', zeroline=True, zerolinecolor='#475569'),
        plot_bgcolor='#0f172a', paper_bgcolor='#1e293b',
        font=dict(color='#e2e8f0'), height=400,
        updatemenus=[{'type':'buttons','showactive':False,
            'buttons':[{'label':'▶  Reveal','method':'animate',
                'args':[None,{'frame':{'duration':1200,'redraw':True},
                              'transition':{'duration':600},'fromcurrent':True}]}],
            'x':0.5,'xanchor':'center','y':-0.12,'yanchor':'top'}]))
_fig_reveal.show()


# ── Reveal price chart ───────────────────────────────────────────────────────
display(HTML('<div class="ag-section-hdr">Where your calls sit on the Tesla timeline</div>'))
try:
    display(Image(filename=f'{DRIVE_ROOT}/TSLA_price_timeline_reveal.png', width=900))
except Exception:
    display(HTML('<p style="color:#94a3b8;">Reveal chart not found.</p>'))
# ── Scoring ──────────────────────────────────────────────────────────────────
import time; time.sleep(0.3)
display(HTML('<div class="ag-section-hdr">Scoring</div>'))

_score = 0
_breakdown = []
_pp_q = STATE.get('power_play_quarter')
for _call in CALLS:
    _q = _call['quarter']
    _actual = _call['returns']['direction']
    _pred   = pred_dropdowns[_q].value
    _chips  = STATE['chips'][_q]
    _mul    = 2 if (_q == _pp_q) else 1
    _ok = ((_pred=='Stock rose' and _actual=='positive') or
           (_pred=='Stock fell' and _actual=='negative'))
    _pts = _chips * _mul if _ok else -_chips * _mul
    _score += _pts
    _breakdown.append(dict(call=ANON_MAP.get(_q,_q), actual_q=_q, prediction=_pred, actual=_actual,
                           chips=_chips, mul=_mul, pts=_pts, ok=_ok))

_rows_html = ''
for _b in _breakdown:
    _col = '#22c55e' if _b['ok'] else '#ef4444'
    _act_str = 'positive ↑' if _b['actual']=='positive' else 'negative ↓'
    _pp_badge = ' <span style="color:#f59e0b;font-size:0.85em;">⚡x2</span>' if _b.get('mul',1)==2 else ''
    _rows_html += (
        f'<tr><td>{_b["call"]}{_pp_badge}</td><td>{_b["prediction"]}</td>'
        f'<td>{_act_str}</td><td>{_b["chips"]}</td>'
        f'<td style="color:{_col};">{"+" if _b["ok"] else ""}{_b["pts"]}</td></tr>')
_tot_col = '#22c55e' if _score >= 0 else '#ef4444'
display(HTML(
    f'<table class="ag-table">'
    f'<tr><th>Call</th><th>Your prediction</th><th>Actual</th>'
    f'<th>Chips</th><th>Points</th></tr>'
    f'{_rows_html}'
    f'<tr style="border-top:2px solid #334155;">'
    f'<td colspan="4" style="text-align:right;color:#94a3b8;">Total</td>'
    f'<td style="color:{_tot_col};font-weight:bold;">{"+" if _score>=0 else ""}{_score}</td></tr>'
    f'</table>'))

# ── Upset calls ───────────────────────────────────────────────────────────────
_upset = [c['quarter'] for c in CALLS
          if (c['excerpt']['musk_net_sent']>0.01 and c['returns']['direction']=='negative')
          or (c['excerpt']['musk_net_sent']<-0.01 and c['returns']['direction']=='positive')]
if _upset:
    display(HTML(
        '<div style="background:#1e293b;border:1px solid #f59e0b;border-radius:8px;'
        'padding:14px 20px;margin:12px 0;font-family:Georgia;">'
        '<p style="color:#f59e0b;margin:0 0 8px 0;font-weight:bold;">'
        f'Upset calls: {", ".join(_upset)}</p>'
        '<p style="color:#e2e8f0;font-size:0.9em;">Calls where Musk\'s LM sentiment '
        'pointed in the opposite direction to the actual abnormal return. '
        'These are where acoustic and visual signals carry the most information value.</p>'
        '</div>'))

# ── Debrief matrix ────────────────────────────────────────────────────────────
display(HTML('<div class="ag-section-hdr">Debrief matrix — signal coherence</div>'))

_matrix_rows = ''
for _call in CALLS:
    _q      = _call['quarter']
    _net    = _call['excerpt']['musk_net_sent']
    _lm_dir = 'positive' if _net>0.005 else ('negative' if _net<-0.005 else 'neutral')
    _row    = SUMMARY[SUMMARY['call']==f'TSLA_{_q}']
    _fou    = _row['fraction_of_unvoiced_mean_musk'].values
    _aco    = ('negative' if (len(_fou) and not np.isnan(_fou[0]) and _fou[0]>FOU_MEAN)
               else 'positive' if len(_fou) else 'n/a')
    _actual = _call['returns']['direction']
    def _bg(sig, actual=_actual):
        if sig in ('neutral','n/a'): return '#1e293b'
        return '#14532d' if sig==actual else '#450a0a'
    def _arr(sig):
        if sig=='n/a':       return '<span class="ag-na">n/a</span>'
        if sig=='positive':  return '↑ positive'
        if sig=='neutral':   return '→ neutral'
        return '↓ negative'
    _act_col = '#22c55e' if _actual=='positive' else '#ef4444'
    _matrix_rows += (
        f'<tr><td style="color:#818cf8;font-weight:bold;">{_q}</td>'
        f'<td style="background:{_bg(_lm_dir)};">{_arr(_lm_dir)}</td>'
        f'<td style="background:{_bg(_aco)};">{_arr(_aco)}</td>'
        f'<td style="background:#1e293b;"><span class="ag-na">n/a</span></td>'
        f'<td style="color:{_act_col};font-weight:bold;">{_arr(_actual)}</td></tr>')

display(HTML(
    '<p style="color:#94a3b8;font-size:0.85em;margin:4px 0 8px 0;">'
    'Green cell = signal matched outcome. Red = contradicted. '
    'Images column shows n/a (EDGAR HTML cover photos only).</p>'
    '<table class="ag-table">'
    '<tr><th>Call</th><th>Text (LM)</th><th>Audio (FoU)</th>'
    '<th>Images</th><th>Actual outcome</th></tr>'
    f'{_matrix_rows}'
    '</table>'))


<div style='background:linear-gradient(90deg,#1e1b4b,#0f172a);padding:20px 26px;border-radius:8px;border-left:4px solid #94a3b8;margin:20px 0 12px 0;font-family:Georgia,serif;'>
<h2 style='color:#94a3b8;margin:0 0 8px 0;'>Individual Reflection</h2>
<p style='color:#e2e8f0;margin:0 0 10px 0;font-size:0.95em;line-height:1.7;'>
  Answer the three questions below individually &#8212; these are <strong>not shared with your team</strong> and do not affect the leaderboard. Be specific: reference the calls, features, and evidence you actually used today.
</p>
<p style='color:#94a3b8;margin:0;font-size:0.88em;line-height:1.6;'>
  Your responses are submitted directly to the session spreadsheet under your team name. There is no word limit, but aim for two or three substantive sentences per question. Vague answers will not help you in the module assessment.
</p>
</div>

In [ ]:
# @title 💬 Reflections {display-mode: "form"}
_questions = [
    'Which modality (text, images, or audio) was most useful for predicting outcomes, and why?',
    'Describe a call where the three modalities sent conflicting signals. How did you resolve the conflict?',
    'If you were building an automated signal-extraction system for earnings calls, '
    'which single feature would you prioritise and why?',
]
_ref_areas = [widgets.Textarea(placeholder=q,
    layout=widgets.Layout(width='720px', height='90px')) for q in _questions]

ref_submit_btn = widgets.Button(description='Submit reflection',
    layout=widgets.Layout(width='200px'))
ref_status = widgets.HTML()

def on_ref_submit(b):
    if not STATE.get('team_name'):
        ref_status.value='<span style="color:#ef4444;">Register first.</span>'; return
    payload = {'action':'submit_reflection','team_name':STATE['team_name'],
               'q1':_ref_areas[0].value.strip(),
               'q2':_ref_areas[1].value.strip(),
               'q3':_ref_areas[2].value.strip()}
    try:
        resp = post_to_script(payload)
        if resp.get('success'):
            ref_submit_btn.disabled=True
            ref_status.value='<span style="color:#22c55e;">✓ Reflection submitted.</span>'
        else:
            ref_status.value=f'<span style="color:#ef4444;">{resp}</span>'
    except Exception:
        ref_submit_btn.disabled=True
        ref_status.value='<span style="color:#f59e0b;">⚠ Offline — thank you.</span>'

ref_submit_btn.on_click(on_ref_submit)

for _i, (_qlbl, _w) in enumerate(zip(_questions, _ref_areas), 1):
    display(HTML(f'<div style="font-family:Georgia;color:#e2e8f0;margin:12px 0 4px 0;">'
                 f'{_i}. {_qlbl}</div>'))
    display(_w)
display(widgets.HBox([ref_submit_btn, ref_status]))
